In [ ]:
#Install a few packages that aren’t automatically included in Colab
!pip install windrose cartopy cmocean

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 14.8 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import warnings
from datetime import datetime, timedelta
from matplotlib import pyplot as plt
import numpy as np
from windrose import WindroseAxes
import math
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import xarray as xr
import cmocean
import calendar
import geopy.distance
from matplotlib.colors import ListedColormap
import seaborn as sns
import statsmodels.api as sm
import sklearn.metrics as skmet
import matplotlib.patches as mpatches
import re
from google.colab import drive

%matplotlib inline
warnings.filterwarnings("ignore")

drive.mount('/content/drive')

main_dir = '/content/drive/MyDrive/TOR_ETC_Analysis' #path to main location to store data and plots **NOTE: add shortcut from shared folder to your MyDrive
data_dir = os.path.join(main_dir, 'data')
plots_dir = os.path.join(main_dir, 'plots')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/My Drive/TOR_ETC_Analysis" # make sure main_dir is present and accessible

# Mkdirs and process data

In [ ]:
# mkdirs if they don't already exist
if not os.path.exists(main_dir):
    os.mkdir(main_dir)

if not os.path.exists(data_dir):
    os.mkdir(data_dir)

if not os.path.exists(plots_dir):
    os.mkdir(plots_dir)

# Files needed to be downloaded / saved to data_dir
# From: https://www.spc.noaa.gov/wcm/#data
    # File: 1950-2022_actual_tornadoes.csv
# From: Michelle Gore
    # File: trajsens2.ERA5.hourly.txt
# From: https://www.spc.noaa.gov/exper/tctor/
    # Filter on F1/EF1+ and copy data to a spreadsheet to save as CSV (also provided from Lauren)

In [ ]:
%%time
# Read in tornado data
tor_file = '%s/1950-2022_actual_tornadoes.csv' % data_dir
if os.path.exists(tor_file):
    tor_data = pd.read_csv(tor_file, header=[0])
    tor_data = tor_data[tor_data['yr'].apply(lambda x: int(x) >= 1980)]
    tor_data = tor_data.sort_values(by=['date', 'time'])
    tor_data = tor_data.drop(tor_data[tor_data.mag.isin([0, -9])].index)

    # Store UTCs to add to dataframe
    utcyr = []
    utcmo = []
    utcdy = []
    utctime = []
    utcdate = []

    # Convert times to UTC
    for index, row in tor_data.iterrows():
        year = tor_data['yr'][index]
        month = tor_data['mo'][index]
        day = tor_data['dy'][index]
        time = tor_data['time'][index].split(":")
        hour = time[0]
        min = time[1]
        timezone = tor_data['tz'][index]
        time_diff = 6  # all are tz=3 (CST), except for five cases where tz=6 (these are removed later)
        date = datetime(int(year), int(month), int(day), int(hour), int(min))
        time_diff_date = date + timedelta(hours=time_diff)

        utc_year = str(time_diff_date.year)
        utc_month = str(format(time_diff_date.month)).zfill(2)
        utc_day = str(format(time_diff_date.day)).zfill(2)
        utc_hour = str(format(time_diff_date.hour)).zfill(2)
        utc_min = str(format(time_diff_date.minute)).zfill(2)

        utcyr.append(int(utc_year))
        utcmo.append(int(utc_month))
        utcdy.append(int(utc_day))
        utctime.append(utc_hour + ":" + utc_min + ":00")
        utcdate.append(utc_year + "-" + utc_month + "-" + utc_day)

    # Add UTC date and time to dataframe
    tor_data['utcyr'] = utcyr
    tor_data['utcmo'] = utcmo
    tor_data['utcdy'] = utcdy
    tor_data['utcdate'] = utcdate
    tor_data['utctime'] = utctime

tor_data.to_csv('%s/1980-2022_actual_tornadoes.csv' % data_dir, index=False)

CPU times: user 4.47 s, sys: 106 ms, total: 4.58 s
Wall time: 6.77 s


In [ ]:
%%time

# THIS TAKES ABOUT 4 HOURS TO RUN (file created is provided by Lauren)

# Add ETC data
tor_file = '%s/1980-2022_actual_tornadoes.csv' % data_dir
if os.path.exists(tor_file):
    tor_data = pd.read_csv(tor_file, header=[0])

    etc_lat = []
    etc_lon = []
    etc_slp = []

    for index, row in tor_data.iterrows():
        year = str(tor_data['utcyr'][index])
        month = str(tor_data['utcmo'][index])
        day = str(tor_data['utcdy'][index])
        hour = tor_data['utctime'][index][:2]
        tor_date = datetime(int(year), int(month), int(day), int(hour))
        print(tor_date)

        # Read in ETC tracking data
        filename = "%s/trajsens2.ERA5.hourly.txt" % data_dir
        etc_df = pd.DataFrame(columns=['lat', 'lon', 'slp'])

        if os.path.exists(filename):
            file = open(filename, 'r')
            # Read through file line by line
            while True:
                line = file.readline()
                if not line:
                    break
                else:
                    linesplit = line.strip().split('\t')
                    if not linesplit[0].startswith("start") and (year == linesplit[6]) and (month == linesplit[7]) and (day == linesplit[8]) and (hour == linesplit[9].zfill(2)):
                        df = pd.DataFrame({"lat": [linesplit[3]], "lon": [linesplit[2]], "slp": [linesplit[4]]})
                        etc_df = etc_df._append(df, ignore_index=True)

            file.close()

        etc_df.loc[:, "lon"] = etc_df["lon"].apply(lambda x: float(x) - 360 if float(x) > 180 else float(x))
        etc_df.loc[:, "lat"] = etc_df["lat"].apply(lambda x: float(x))

        etc_df = etc_df.loc[(etc_df["lon"] <= -55) & (etc_df["lon"] >= -135) & (etc_df["lat"] <= 70) & (etc_df["lat"] >= 20)]

        etc_lat.append(etc_df['lat'].values)
        etc_lon.append(etc_df['lon'].values)
        etc_slp.append(etc_df['slp'].values)

    tor_data['etclats'] = etc_lat
    tor_data['etclons'] = etc_lon
    tor_data['etcslp'] = etc_slp

tor_data.to_csv('%s/1980-2022_tornadoes_and_etcs.csv' % data_dir, index=False)

Streaming output truncated to the last 5000 lines.
2004-05-25 04:00:00
2004-05-25 04:00:00
2004-05-25 04:00:00
2004-05-25 05:00:00
2004-05-25 21:00:00
2004-05-25 21:00:00
2004-05-26 03:00:00
2004-05-26 21:00:00
2004-05-26 21:00:00
2004-05-27 01:00:00
2004-05-27 02:00:00
2004-05-27 03:00:00
2004-05-27 06:00:00
2004-05-27 20:00:00
2004-05-27 22:00:00
2004-05-27 22:00:00
2004-05-27 23:00:00
2004-05-27 23:00:00
2004-05-28 00:00:00
2004-05-28 00:00:00
2004-05-28 00:00:00
2004-05-28 02:00:00
2004-05-28 03:00:00
2004-05-29 21:00:00
2004-05-29 22:00:00
2004-05-29 22:00:00
2004-05-29 23:00:00
2004-05-29 23:00:00
2004-05-29 23:00:00
2004-05-29 23:00:00
2004-05-29 23:00:00
2004-05-30 00:00:00
2004-05-30 00:00:00
2004-05-30 00:00:00
2004-05-30 00:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 01:00:00
2004-05-30 02:00:00
2004-05-3

In [ ]:
%%time
# Add in date-time column and find closest ETC track and distance
tor_file = '%s/1980-2022_tornadoes_and_etcs.csv' % data_dir
tor_data = pd.read_csv(tor_file, header=[0])
date_times = []
close_etclat = []
close_etclon = []
distances = []
pressures = []
for index, row in tor_data.iterrows():
    date = tor_data['utcdate'][index]
    time = tor_data['utctime'][index][:2]
    date_time = date + " " + time + ":00:00"
    date_times.append(date_time)

    torlat = tor_data['slat'][index]
    torlon = tor_data['slon'][index]
    etclats = tor_data['etclats'][index]
    etclons = tor_data['etclons'][index]
    tor_coord = (torlat, torlon)
    etcslps = tor_data['etcslp'][index]

    # If empty: no ETC present
    if etclats == '[]':
        closest_lat = 'nan'
        closest_lon = 'nan'
        closest_distance = '-1'
        closest_pressure = 'nan'
    else:
        # Convert string to list
        # Step 1: Remove square brackets
        etclats_str = etclats.strip('[]')
        etclons_str = etclons.strip('[]')
        etcslps_str = etcslps.strip('[]')
        # Step 2: Split the string by space
        etclats_str = re.sub(r'\s+', ' ', etclats_str).strip()
        etclons_str = re.sub(r'\s+', ' ', etclons_str).strip()
        etcslps_str = re.sub(r'\s+', ' ', etcslps_str).strip()
        etclats_values = etclats_str.split(' ')
        etclons_values = etclons_str.split(' ')
        etcslps_values = etcslps_str.split(' ')
        # Step 3: Convert to floats
        etclats_list = [float(value) for value in etclats_values]
        etclons_list = [float(value) for value in etclons_values]
        etcslps_list = [str(value) for value in etcslps_values]

        if len(etclats_list) > 1:
            i = 0
            # find the closest track
            closest_distance = 10000000
            while i < len(etclats_list):
                etc_coord = (float(etclats_list[i]), float(etclons_list[i]))
                distance = geopy.distance.geodesic(tor_coord, etc_coord).km
                if distance < closest_distance:
                    closest_distance = distance
                    closest_lat = etclats_list[i]
                    closest_lon = etclons_list[i]
                    closest_pressure = etcslps_list[i]
                i = i + 1
        else:
            closest_lat = etclats_list[0]
            closest_lon = etclons_list[0]
            closest_pressure = etcslps_list[0]
            etc_coord = (float(etclats_list[0]), float(etclons_list[0]))
            closest_distance = geopy.distance.geodesic(tor_coord, etc_coord).km

    distances.append(closest_distance)
    pressures.append(closest_pressure)
    close_etclat.append(closest_lat)
    close_etclon.append(closest_lon)

tor_data['utcdatetimes'] = date_times
tor_data['closest_etclat'] = close_etclat
tor_data['closest_etclon'] = close_etclon
tor_data['distance'] = distances
tor_data['pressure'] = pressures

tor_data.to_csv('%s/1980-2022_tornadoes_and_etcs.csv' % data_dir, index=False)

CPU times: user 13.9 s, sys: 85.2 ms, total: 14 s
Wall time: 14.2 s


In [ ]:
%%time
# Remove tropical cyclone tornadoes
tor_file = '%s/1980-2022_tornadoes_and_etcs.csv' % data_dir
tctor_file = '%s/TropCycTornadoes.csv' % data_dir

tor_df = pd.read_csv(tor_file, header=[0])
tctor_df = pd.read_csv(tctor_file, header=[0])
orig_len = int(len(tor_df))
same_tor = 0
for index, row in tctor_df.iterrows():
    ef = tctor_df['F or EF'][index]

    start_latlon = tctor_df['Start Lat/Lon'][index]
    start_latlon = start_latlon.split('/')
    start_lat = float(start_latlon[0])
    start_lon = float(start_latlon[1])

    date_str = tctor_df['Date/Time (UTC)'][index]
    date_str = date_str.split(' ')
    minute = date_str[1][2:]
    time = datetime.strptime('%s00' % date_str[1][:2], '%H%M').time()
    date = datetime.strptime(date_str[0], '%Y/%m/%d')
    tctor_date = datetime.combine(date, time)

    year = tctor_date.year
    month = str(tctor_date.month).zfill(2)
    day = str(tctor_date.day).zfill(2)
    time = str(tctor_date.hour).zfill(2)

    # Matching tornado data with TCTOR
    year_df = tor_df[tor_df['utcyr'].isin([int(year)])]
    month_df = year_df[year_df['utcmo'].isin([int(month)])]
    day_df = month_df[(month_df['utcdy'].isin([int(day)]))]
    tor_hour_df = day_df[(day_df['utctime'].apply(lambda x: x[:2] == time))]

    # Removing any tornado that occurs in the same hour and within 2.5 degrees of a TCTOR
    if not tor_hour_df.empty:
        begin_lat = tor_hour_df[tor_hour_df['slat'].apply(lambda x: (start_lat - 2.5) <= float("{:.2f}".format(x)) <= (start_lat + 2.5))]
        if not begin_lat.empty:
            begin_lon = begin_lat[begin_lat['slon'].apply(lambda x: (start_lon - 2.5) <= float("{:.2f}".format(x)) <= (start_lon + 2.5))]
            if not begin_lon.empty:
                tor_df.drop(begin_lon.index, inplace=True)
                same_tor = same_tor + 1

tor_df.to_csv('%s/1980-2022_non_tc_tor_and_etcs.csv' % data_dir, index=False)

CPU times: user 4.48 s, sys: 54.6 ms, total: 4.54 s
Wall time: 4.91 s


In [ ]:
%%time
# Remove additional tropical cyclone tornadoes
tor_file = '%s/1980-2022_non_tc_tor_and_etcs.csv' % data_dir
tctor_file = '%s/tornado_data.csv' % data_dir

tor_df = pd.read_csv(tor_file, header=[0])
orig_len = int(len(tor_df))
tctor_df = pd.read_csv(tctor_file, header=[0])
tctor_df = tctor_df.drop(tctor_df[tctor_df.FSCALE.isin([0])].index) # Remove F/EF0s
same_tor = 0
for index, row in tctor_df.iterrows():
    madeit = False
    ef = tctor_df['FSCALE'][index]

    start_lat = float(tctor_df['STLAT'][index])
    start_lon = float(tctor_df['STLON'][index])

    # CST datetime of TC Tor
    year = tctor_df['YEAR'][index]
    month = tctor_df['MON'][index]
    day = tctor_df['DAY'][index]
    time = tctor_df['TIMECST'][index] # b'15:00:00'
    time_str = time.replace('b', '')
    time_str = time_str.replace('\'', '')
    time_str = time_str.split(':')
    hour = time_str[0]
    min = time_str[1]

    time_diff = 6  # all are CST, need to convert to UTC
    tc_tor_date = datetime(int(year), int(month), int(day), int(hour), int(min))
    tc_tor_utc_date = tc_tor_date + timedelta(hours=time_diff)

    utcyear = tc_tor_utc_date.year
    utcmonth = str(tc_tor_utc_date.month).zfill(2)
    utcday = str(tc_tor_utc_date.day).zfill(2)
    utchour = str(tc_tor_utc_date.hour).zfill(2)
    utcminute = str(tc_tor_utc_date.minute).zfill(2)

    # Matching tornado data with TCTOR
    year_df = tor_df[tor_df['utcyr'].isin([int(utcyear)])]
    month_df = year_df[year_df['utcmo'].isin([int(utcmonth)])]
    day_df = month_df[(month_df['utcdy'].isin([int(utcday)]))]
    if not (day_df.empty):
      hour_df = day_df[(day_df['utctime'].apply(lambda x: x[:2] == utchour))]
      #min_df = hour_df[(hour_df['utctime'].apply(lambda x: x[3:5] == utcminute))]

      # Removing any tornado that occurs in the same hour and within 2.5 degrees of a TCTOR
      if not hour_df.empty:
          begin_lat = hour_df[hour_df['slat'].apply(lambda x: (start_lat - 2.5) <= float("{:.2f}".format(x)) <= (start_lat + 2.5))]
          if not begin_lat.empty:
              begin_lon = begin_lat[begin_lat['slon'].apply(lambda x: (start_lon - 2.5) <= float("{:.2f}".format(x)) <= (start_lon + 2.5))]
              if not begin_lon.empty:
                  tor_df.drop(begin_lon.index, inplace=True)

print()
print(orig_len)
print(len(tor_df))
print(orig_len  - len(tor_df))
print()
tor_df.to_csv('%s/1980-2022_add_non_tc_tor_and_etcs.csv' % data_dir, index=False)


20679
20530
149

CPU times: user 2.17 s, sys: 25.4 ms, total: 2.2 s
Wall time: 2.25 s


In [ ]:
%%time
# Remove tornadoes with tz != 3
# Dataset description says all timezones were converted to 3=CST
# However, there are 5 tornadoes still in our data that have tz=6
tor_file = '%s/1980-2022_non_tc_tor_and_etcs.csv' % data_dir
if os.path.exists(tor_file):
    tor_data = pd.read_csv(tor_file, header=[0])
    tor_data = tor_data[tor_data['tz'].isin([3])]
    tor_data.to_csv('%s/1980-2022_non_tc_tor_and_etcs.csv' % data_dir, index=False)

CPU times: user 718 ms, sys: 31.7 ms, total: 749 ms
Wall time: 829 ms
